# Phishing URL Detection: PhishTransformer

This notebook compares character-level CNN-only, Transformer-only, and hybrid CNN + Transformer models. Reported metrics are generated only after the supplied datasets are loaded and the cells are run.

## 1. Introduction and Base Paper Technique

The base paper is Asiri et al., *PhishTransformer: A Novel Approach to Detect Phishing Attacks Using URL Collection and Transformer*, Electronics 13(1), 30 (2024), DOI: 10.3390/electronics13010030. It collects URLs embedded in webpage content, including hyperlinks and iframes, then combines convolutional feature extraction with Transformer encoders. This project adapts that idea to URL-string-only classification because the available UNB CIC URL-2016, PhishTank, and Tranco sources do not include page HTML. This is an adaptation, not a claim of reproducing the paper's page-content pipeline.

## 2. Dataset Loading

Expected files: `Dataset/All.csv`, `Dataset/verified_online.csv`, and `Dataset/top-1m.csv`. The datasets are not distributed with this repository.

In [1]:
import json, re, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import callbacks, layers, losses, metrics, models, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
MAX_LEN = 256
EMBED_DIM = 128
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

I0000 00:00:1791470804.318115    3731 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791470804.443141    3731 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791470810.425986    3731 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
def infer_url_col(frame):
    candidates = [c for c in frame.columns if any(k in c.lower() for k in ('url','link','domain','website','site'))]
    return candidates[0] if candidates else frame.columns[0]

def load_phishing_csv(path):
    frame = pd.read_csv(path, dtype=str, low_memory=False)
    return pd.DataFrame({'url': frame[infer_url_col(frame)], 'label': 1})

dataset_dir = Path('Dataset')
df_all = load_phishing_csv(dataset_dir / 'All.csv')
df_verified = load_phishing_csv(dataset_dir / 'verified_online.csv')
top = pd.read_csv(dataset_dir / 'top-1m.csv', header=None, names=['rank','domain'], dtype=str)
df_top = pd.DataFrame({'url': 'http://' + top['domain'].str.strip().str.lower(), 'label': 0})
data = pd.concat([df_all, df_verified, df_top], ignore_index=True)
print(data['label'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset/All.csv'

## 3. Data Cleaning

In [ ]:
def clean_url(value):
    value = re.sub(r'\s+', '', str(value).strip().lower())
    return value if value.startswith(('http://','https://')) else 'http://' + value

data['url'] = data['url'].dropna().map(clean_url)
data = data.drop_duplicates('url').reset_index(drop=True)
data = data[data['url'].str.len() > 0].copy()
print(f'Clean URLs: {len(data):,}')

## 4. Exploratory Data Analysis

In [ ]:
eda = data.assign(length=data['url'].str.len())
display(eda.groupby('label')['length'].describe())
sns.countplot(data=data, x='label')
plt.title('Class distribution before balancing')
plt.xlabel('Label (0 = legitimate, 1 = phishing)')
plt.show()

## 5. Character-level URL Encoding

In [ ]:
chars = sorted(set(''.join(data['url'].tolist())))
vocab = ['<PAD>', '<UNK>'] + [char for char in chars if char not in ('<PAD>', '<UNK>')]
char2idx = {char: index for index, char in enumerate(vocab)}
def encode(url, maxlen=MAX_LEN):
    values = [char2idx.get(char, char2idx['<UNK>']) for char in url[:maxlen]]
    return values + [char2idx['<PAD>']] * (maxlen - len(values))
data['sequence'] = data['url'].map(encode)
print(f'Vocabulary: {len(vocab)} characters; max URL length: {MAX_LEN}')

## 6. Data Balancing and 7. Train/Test Split

In [ ]:
class_size = data['label'].value_counts().min()
balanced = pd.concat([
    data[data['label'] == label].sample(class_size, random_state=RANDOM_STATE)
    for label in (0, 1)